In [2]:
# 設定路徑

from pathlib import Path
import matplotlib.pyplot as plt

TABLE_DIR = Path("output/table")
FIGURE_DIR = Path("output/figure")

TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)


def save_table(df, filename):
    """將 pandas DataFrame 存成 CSV。"""
    path = TABLE_DIR / filename
    df.to_csv(path, index=False, encoding="utf-8-sig")
    return path


def save_figure(filename, fig=None):
    """將 Matplotlib 圖片存成檔案。"""
    path = FIGURE_DIR / filename
    (fig or plt.gcf()).savefig(path, bbox_inches="tight", dpi=300)
    return path

In [3]:
# import packages
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


In [4]:
# import datasets

df_customers = pd.read_csv("raw_data/olist_customers_dataset.csv")
df_orders = pd.read_csv("raw_data/olist_orders_dataset.csv")
df_order_items = pd.read_csv("raw_data/olist_order_items_dataset.csv")
df_products = pd.read_csv("raw_data/olist_products_dataset.csv")
df_sellers = pd.read_csv("raw_data/olist_sellers_dataset.csv")
df_product_category_name_translation = pd.read_csv("raw_data/product_category_name_translation.csv")
df_geolocation = pd.read_csv("raw_data/olist_geolocation_dataset.csv")
df_order_payments = pd.read_csv("raw_data/olist_order_payments_dataset.csv")
df_order_reviews = pd.read_csv("raw_data/olist_order_reviews_dataset.csv")

- 商品類別為葡萄牙文，先合併翻譯表轉成英文，後續的檢查與分析都使用英文類別。翻譯表缺少 `pc_gamer` 與 `portateis_cozinha_e_preparadores_de_alimentos` 2 個類別，先手動補上

In [5]:
# translate product categories into English

# 1) add the 2 categories missing from the translation table
missing_translation = pd.DataFrame({
    "product_category_name": ["pc_gamer", "portateis_cozinha_e_preparadores_de_alimentos"],
    "product_category_name_english": ["pc_gamer", "portable_kitchen_food_preparers"],
})
df_product_category_name_translation = pd.concat(
    [df_product_category_name_translation, missing_translation], ignore_index=True
)

# 2) merge into df_products
rows_before = len(df_products)
df_products = df_products.merge(df_product_category_name_translation,
                                on="product_category_name", how="left")

# 3) check: row count unchanged, and only the 610 products without a category are left untranslated
print("rows:", rows_before, "->", len(df_products))
print("untranslated:", df_products["product_category_name_english"].isna().sum())

rows: 32951 -> 32951
untranslated: 610


In [6]:
# check the datasets separately
dfs = {
    "customers": df_customers,
    "orders": df_orders,
    "order_items": df_order_items,
    "products": df_products,
    "sellers": df_sellers,
    "category_translation": df_product_category_name_translation,
    "geolocation": df_geolocation,
    "order_payments": df_order_payments,
    "order_reviews": df_order_reviews
}

for name, df in dfs.items():
    print(f"=== {name}: {df.shape[0]:,} rows x {df.shape[1]} cols")
    display(df.head())

=== customers: 99,441 rows x 5 cols


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


=== orders: 99,441 rows x 8 cols


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


=== order_items: 112,650 rows x 7 cols


...

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


=== order_payments: 103,886 rows x 5 cols


,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


=== order_reviews: 99,224 rows x 7 cols


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [7]:
# datasets overall information
summary = pd.DataFrame([
    {"table": name,
     "rows": len(df),
     "cols": df.shape[1],
     "duplicated_rows": df.duplicated().sum(),
     "cols_with_missing": (df.isna().sum() > 0).sum(),
     "total_missing": df.isna().sum().sum()}
    for name, df in dfs.items()
])
summary

,table,rows,cols,duplicated_rows,cols_with_missing,total_missing
0,customers,99441,5,0,0,0
1,orders,99441,8,0,3,4908
2,order_items,112650,7,0,0,0
3,products,32951,10,0,9,3058
4,sellers,3095,4,0,0,0
5,category_translation,73,2,0,0,0
6,geolocation,1000163,5,261831,0,0
7,order_payments,103886,5,0,0,0
8,order_reviews,99224,7,0,2,145903


- 針對orders、products各別檢查並處理missing values，order_reviews暫時用不到因此先跳過

#### 檢查 & 處理缺失值

In [8]:
# check missing values in df_orders
df_orders.isna().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [9]:
# check the order status for these three columns with missing values in df_orders, and see if there is any pattern
date_cols = ["order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date"]
for col in date_cols:
    print(f"=== {col}: {df_orders[col].isna().sum()} missing")
    print(df_orders.loc[df_orders[col].isna(), "order_status"].value_counts(), "\n")

=== order_approved_at: 160 missing
order_status
canceled     141
delivered     14
created        5
Name: count, dtype: int64 

=== order_delivered_carrier_date: 1783 missing
order_status
unavailable    609
canceled       550
invoiced       314
processing     301
created          5
approved         2
delivered        2
Name: count, dtype: int64 

=== order_delivered_customer_date: 2965 missing
order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64 



- orders 有 3 個日期欄位存在缺值。交叉比對訂單狀態後發現，絕大多數是取消或尚未送達的訂單，屬於合理缺值，因此保留不刪除，也不填補。另有少量矛盾資料（例如已送達卻無送達日期、送達早於出貨），合計不到 0.3%，在計算配送指標時排除。

In [10]:
# check missing values in df_products
df_products.isna().sum()

product_id                         0
product_category_name            610
product_name_lenght              610
product_description_lenght       610
product_photos_qty               610
product_weight_g                   2
product_length_cm                  2
product_height_cm                  2
product_width_cm                   2
product_category_name_english    610
dtype: int64

- 從結果來看，商品類別名稱有缺值的共610個，name_lenght、description_lenght、photos_qty等欄位也相依缺610個，惟後續分析僅需使用商品類別，因此只處理product_category_name即可

- 缺類別的商品仍有實際銷售，因此不刪除，英文類別填入 `unknown`

In [11]:
# fill missing categories with "unknown"
df_products["product_category_name_english"] = df_products["product_category_name_english"].fillna("unknown")

print("missing:", df_products["product_category_name_english"].isna().sum())
df_products["product_category_name_english"].value_counts().head(10)

missing: 0


product_category_name_english
bed_bath_table           3029
sports_leisure           2867
furniture_decor          2657
health_beauty            2444
housewares               2335
auto                     1900
computers_accessories    1639
toys                     1411
watches_gifts            1329
telephony                1134
Name: count, dtype: int64

#### 檢查資料類型 (data type)

In [12]:
for name, df in dfs.items():
    print(f"=== {name}")
    df.info()
    print()

=== customers
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 3.8 MB

=== orders
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_appr

- orders、order_items、order_reviews共有八個時間屬性欄位目前都是str type，須先轉成date

In [13]:
# convert date columns from str to datetime
date_columns = {
    "orders": ["order_purchase_timestamp", "order_approved_at",
               "order_delivered_carrier_date", "order_delivered_customer_date",
               "order_estimated_delivery_date"],
    "order_items": ["shipping_limit_date"],
    "order_reviews": ["review_creation_date", "review_answer_timestamp"],
}

for name, cols in date_columns.items():
    df = dfs[name]
    for col in cols:
        df[col] = pd.to_datetime(df[col])

# check: these columns should now be datetime64
for name, cols in date_columns.items():
    print(f"=== {name}")
    print(dfs[name][cols].dtypes, "\n")

=== orders
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object 

=== order_items
shipping_limit_date    datetime64[us]
dtype: object 

=== order_reviews
review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
dtype: object 



In [ ]:
# export cleaned tables for further analysis
PROCESSED_DIR = Path("processed_data")
PROCESSED_DIR.mkdir(exist_ok=True)

export_tables = {
    "orders": df_orders,
    "order_items": df_order_items,
    "customers": df_customers,
    "products": df_products,
    "order_reviews": df_order_reviews.drop(columns=["review_comment_title", "review_comment_message"]),
}

for name, df in export_tables.items():
    df.to_csv(PROCESSED_DIR / f"{name}.csv", index=False, na_rep="\\N", lineterminator="\n")
    print(f"{name:14s} {len(df):>7,} rows -> processed_data/{name}.csv")

orders          99,441 rows -> processed_data/orders.csv
order_items    112,650 rows -> processed_data/order_items.csv
customers       99,441 rows -> processed_data/customers.csv
products        32,951 rows -> processed_data/products.csv
order_reviews   99,224 rows -> processed_data/order_reviews.csv
